# Customer Support Ticket Classification & Priority Tagging System
**Future Interns — Machine Learning Task 2**  
**Author:** Lohith Gopi  

### Objectives
1. **Text Preprocessing & Cleaning:** Normalize ticket text, strip noise, tokenize.
2. **Category Classification:** Route tickets to Technical Support, Billing & Payments, Customer Service, Outages, or Internal / HR.
3. **Priority Tagging (High / Medium / Low):** Predict urgency and apply business SLA escalation rules.
4. **Performance Evaluation:** Measure precision, recall, F1, accuracy, and confusion matrices.

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Load dataset
df = pd.read_csv('data/tickets.csv')
print('Dataset shape:', df.shape)
df.head()

## 1. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=df, y='category', order=df['category'].value_counts().index, ax=axes[0], palette='Blues_r', hue='category', legend=False)
axes[0].set_title('Ticket Category Distribution')

sns.countplot(data=df, x='priority', order=['low', 'medium', 'high'], ax=axes[1], palette=['#2ecc71', '#f39c12', '#e74c3c'], hue='priority', legend=False)
axes[1].set_title('Ticket Priority Distribution')
plt.tight_layout()
plt.show()

## 2. Text Preprocessing & Cleaning

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['combined_text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
df['cleaned_sample'] = df['combined_text'].apply(clean_text)
df[['subject', 'cleaned_sample']].head(3)

## 3. Pipeline Training (Category & Priority)

In [ ]:
X = df['combined_text']
y_cat = df['category']
y_prio = df['priority']

X_train, X_test, y_c_train, y_c_test, y_p_train, y_p_test = train_test_split(
    X, y_cat, y_prio, test_size=0.2, random_state=42, stratify=y_cat
)

# Category Pipeline
cat_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text, stop_words='english', max_features=10000, sublinear_tf=True, ngram_range=(1, 2))),
    ('clf', LinearSVC(class_weight='balanced', random_state=42))
])
cat_pipeline.fit(X_train, y_c_train)

# Priority Pipeline
prio_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text, stop_words='english', max_features=10000, sublinear_tf=True, ngram_range=(1, 2))),
    ('clf', LinearSVC(class_weight='balanced', random_state=42))
])
prio_pipeline.fit(X_train, y_p_train)
print('Both pipelines trained successfully.')

## 4. Evaluation Reports

In [ ]:
c_preds = cat_pipeline.predict(X_test)
p_preds = prio_pipeline.predict(X_test)

print('--- Category Report ---')
print(classification_report(y_c_test, c_preds, digits=3))

print('--- Priority Report ---')
print(classification_report(y_p_test, p_preds, digits=3))

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

c_labels = np.unique(y_c_test)
sns.heatmap(confusion_matrix(y_c_test, c_preds, labels=c_labels), annot=True, fmt='d', cmap='Blues', xticklabels=c_labels, yticklabels=c_labels, ax=axes[0])
axes[0].set_title('Category Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

p_labels = ['low', 'medium', 'high']
sns.heatmap(confusion_matrix(y_p_test, p_preds, labels=p_labels), annot=True, fmt='d', cmap='Oranges', xticklabels=p_labels, yticklabels=p_labels, ax=axes[1])
axes[1].set_title('Priority Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## 6. Live Ticket Triage & SLA Priority Escalation

In [ ]:
URGENT_KEYWORDS = {'outage', 'downtime', 'crash', 'crashed', 'breach', 'unauthorized', 'emergency', 'urgent', 'asap', 'offline'}

def triage(subject: str, body: str):
    txt = f'{subject} {body}'
    cat = cat_pipeline.predict([txt])[0]
    raw_p = prio_pipeline.predict([txt])[0]
    urgent = any(k in clean_text(txt).split() for k in URGENT_KEYWORDS)
    final_p = 'high' if urgent else raw_p
    return {'Subject': subject, 'Assigned Category': cat, 'Priority': final_p.upper(), 'SLA Escalation': urgent and raw_p != 'high'}

samples = [
    ('Production Down', 'All customer transactions failing with timeout 504 on payment gateway.'),
    ('Subscription Invoice', 'Can you send the receipt for the March renewal charge?'),
    ('Feature Query', 'Does your platform support webhook export to Slack?')
]
pd.DataFrame([triage(s, b) for s, b in samples])